## 1  Step 2: WINDEL Calculation of Breast Cancer Cohorts

In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
from collections import OrderedDict
import statistics as st
import itertools
import copy
import sys
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# 1.1  Function to create the bins and labels required to bin the data before comparison.

The first argument is how many days back the bins need to go. The second argument is the width of each bin. The function returns two lists: one with the cuts of each bin, and the second with the name or label of each bin.

In [ ]:
def bins_n_labels(time_range, bin_size):
    bins = []
    labels = []
    n = -time_range
    while n <= 0:
        bins.append(n)
        label = f'{n},'
        n += bin_size
        label += f'{n}'
        labels.append(label)
    return bins, labels[:-1]

bins, labels = bins_n_labels(15000,100)

# 1.2  Bring in the Pre-Vivor cohort of breast cancer patients

This cohort was previously determined and saved to the workspace bucket to include breast cancer patients who had a higher risk, either familial or genetic, of getting breast cancer.

In [ ]:
name_of_file_in_bucket = 'control_group_malignant.csv'

my_bucket = os.getenv('WORKSPACE_BUCKET')
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')

previvors_dataframe = pd.read_csv(name_of_file_in_bucket)
previvors_dataframe.head()

1.2.1  Bin the Pre-Vivor events

In [ ]:
previvors_dataframe['bin'] = pd.cut(previvors_dataframe['day_diff'],labels=labels,bins=bins)
previvors_dataframe.head()
previvors_dataframe = previvors_dataframe[pd.notna(previvors_dataframe['bin'])]
previvors_dataframe['concept_code'] = previvors_dataframe['standard_concept_code']
previvors_dataframe.head()

1.2.2  Filter the Pre-Vivor cohort to only include people who have at least 100 events along their timeline

In [ ]:
counts_df = previvors_dataframe.groupby('person_id').size().reset_index(name='count')
filtered_pv_counts = counts_df[counts_df['count'] > 100]
# filtered_pv_counts = counts_df[(counts_df['count'] > 100) & (counts_df['count'] < 1000)]
pre_ids = filtered_pv_counts['person_id']
filtered_previvors = previvors_dataframe[previvors_dataframe['person_id'].isin(pre_ids)]
print(f'{len(filtered_previvors["person_id"].unique().tolist())} pre-vivors have >100 events and <4899')
# print(f'{len(filtered_previvors["person_id"].unique().tolist())} pre-vivors have >100 events and <1000')
print(max(counts_df['count']))

# 1.3  Bring in Breast Cancer cohort

In [ ]:
name_of_file_in_bucket = 'binned_individuals.csv'

my_bucket = os.getenv('WORKSPACE_BUCKET')
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')

binned_individuals = pd.read_csv(name_of_file_in_bucket)
binned_individuals.head()

1.3.1  Filters the Breast Cancer cohort to only include people who have at least 100 events along their timeline

In [ ]:
counts_ind = binned_individuals.groupby('person_id').size().reset_index(name='count')
filtered_bc_counts = counts_ind[(counts_ind['count'] > 100) & (counts_ind['count'] < 4899)]
# filtered_bc_counts = counts_ind[(counts_ind['count'] > 100)]
ids = filtered_bc_counts['person_id']
filtered_bc_binned = binned_individuals[binned_individuals['person_id'].isin(ids)]
print(f'{len(filtered_bc_binned["person_id"].unique().tolist())} breast cancer patients have >100 events')

1.3.2  Creates a Breast Cancer Cohort to match the Pre-Vivor cohort in number

In [ ]:
total_bc_people = filtered_bc_binned['person_id'].unique().tolist()
num_individuals = len(filtered_previvors["person_id"].unique().tolist())
test_bc_cohort = filtered_bc_binned['person_id'].sample(num_individuals, replace=False).tolist()
excluded = set(test_bc_cohort) | set(pre_ids)
total_bc_people = [person for person in total_bc_people if person not in excluded]

# 1.4  Creates a new cohort of Breast Cancer Patients to compare the other groups to when calculating WINDEL scores

In [ ]:
new_sample = filtered_bc_binned[filtered_bc_binned['person_id'].isin(total_bc_people)]
comparison_cohort = new_sample['person_id'].sample(num_individuals, replace=False).tolist()

# 1.5  Calculate WINDEL Scores for the Pre-Vivor Cohort

In [ ]:
previvors_list = filtered_previvors['person_id'].drop_duplicates().tolist()

indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg = {}

for rando1 in previvors_list:
    for rando2 in comparison_cohort:
        person2_df = filtered_previvors[filtered_previvors['person_id'] == rando1]
        person1_df = filtered_bc_binned[filtered_bc_binned['person_id'] == rando2]

        person1_bin_set = set(person1_df['bin'])
        person2_bin_set = set(person2_df['bin'])

        bin_pairs = np.array([(bin1, bin2) for bin1 in person1_bin_set for bin2 in person2_bin_set], dtype=object)

        Asub_list = {bin1: set(person1_df.loc[person1_df['bin'] == bin1, 'concept_code']) for bin1 in person1_bin_set}
        Bsub_list = {bin2: set(person2_df.loc[person2_df['bin'] == bin2, 'concept_code']) for bin2 in person2_bin_set}

        similarity_scores = np.array([
            len(Asub_list[bin1] & Bsub_list[bin2]) / len(Asub_list[bin1] | Bsub_list[bin2]) if len(Asub_list[bin1] | Bsub_list[bin2]) > 0 else 0.0
            for bin1, bin2 in bin_pairs
        ])

        similarity_matrix = similarity_scores.reshape(len(person1_bin_set), len(person2_bin_set))
        mask = np.ones(similarity_matrix.shape, dtype=bool)
        max_dict_ordered = OrderedDict()
        excluded_bins_to_raw_score = OrderedDict()

        while np.any(mask):
            masked_matrix = np.where(mask, similarity_matrix, -np.inf)
            max_index = np.argmax(masked_matrix)
            max_row, max_col = np.unravel_index(max_index, similarity_matrix.shape)
            max_score = similarity_matrix[max_row, max_col]

            tuple_key = tuple(bin_pairs[max_row * len(person2_bin_set) + max_col])
            max_dict_ordered[tuple_key] = max_score
            excluded_bins_to_raw_score.setdefault(tuple_key, []).append(max_score)

            mask[max_row, :] = False
            mask[:, max_col] = False

        sorted_ordered_dict = OrderedDict(sorted(max_dict_ordered.items(), key=lambda x: x[1], reverse=True))
        indexes2actuallyplot.append(list(sorted_ordered_dict.keys()))
        averages_to_actually_graph = [st.mean(sorted_ordered_dict.values())]

        sorted_bins_to_max_score_dict = OrderedDict(sorted(excluded_bins_to_raw_score.items(), key=lambda x: max(x[1]), reverse=True))
        while sorted_bins_to_max_score_dict:
            bins_from_matrix = list(sorted_bins_to_max_score_dict.keys())
            maxes_average = st.mean([val for values in sorted_bins_to_max_score_dict.values() for val in values])
            averages_with_bins.setdefault((rando1, rando2), []).append({bins_from_matrix[-1]: maxes_average})
            sorted_bins_to_max_score_dict.popitem()

        person1_best_scores = {key[0]: value[0] for key, value in excluded_bins_to_raw_score.items()}
        person2_best_scores = {key[1]: value[0] for key, value in excluded_bins_to_raw_score.items()}

        sorted_by_bins_person1 = OrderedDict(sorted(person1_best_scores.items()))
        sorted_by_bins_person2 = OrderedDict(sorted(person2_best_scores.items()))

        if (rando2, rando1) not in split_avg:
            split_avg[(rando2, rando1)] = []

        while sorted_by_bins_person2:
            closest_to_diagnosis_p2 = next(iter(sorted_by_bins_person2))
            curr_average = st.mean(sorted_by_bins_person2.values())
            split_avg[(rando2, rando1)].append({closest_to_diagnosis_p2: curr_average})
            del sorted_by_bins_person2[closest_to_diagnosis_p2]

In [ ]:
result_previvor = []
comparison_preids = [] # This is storing tuples
# the key is the person, the value is the dictionary of bins to scores
for key, value in split_avg.items():
    person2_bins = {}
    for val in value:
        for k, v in val.items():
            bin = int(k.split(',')[0])
            person2_bins[bin] = v
    scores = {}
    score_list = []
    curr_score = 0
    for j in bins:
        curr_bin = j
        if j not in person2_bins:
            curr_score = curr_score
        else:
            curr_score = person2_bins[j]
        scores[curr_bin] = curr_score
        score_list.append(curr_score)
    max_score = 0
    for item in score_list:
        if item > max_score:
            max_score = item
    if max_score != 1.0:
        result_previvor.append(scores)
        comparison_preids.append(key)

In [ ]:
previvor_id_to_list_of_dictionaries = {}
for person in previvors_list:
    previvor_id_to_list_of_dictionaries[person] = []
    for group_index in range(len(result_previvor)):
        if comparison_preids[group_index][1] == person:
            previvor_id_to_list_of_dictionaries[person].append(result_previvor[group_index])

1.5.1  Plot WINDEL Scores for Pre-Vivor Cohort

In [ ]:
plt.figure(figsize=(20,14))

comparison_functions = {}
apex_x_values = {}  # Optional: store apex x-values by ID
total_previvor_windel = {}

for previd, group in previvor_id_to_list_of_dictionaries.items():
    total_previvor_windel[previd] = []
    i = -1
    for dataset in group:
        i += 1
        x = np.array(list(dataset.keys()))
        y = np.array(list(dataset.values()))

        # Find the cutoff to ignore leading zero values
        index = 0
        while index < len(y) and y[index] == 0:
            index += 1
        y_cutoff = y[index:]
        x_cutoff = x[index:]
        if x_cutoff.size == 0:
            continue

        # Fit a quadratic model
        coefficients = np.polyfit(x_cutoff, y_cutoff, 2)  # coefficients = [a, b, c]
        a, b = coefficients[0], coefficients[1]
        comparison_functions[comparison_preids[i]] = coefficients

        # Create model and predictions
        quadratic_model = np.poly1d(coefficients)
        x_fit = np.linspace(x.min(), x.max(), 100)
        y_fit = quadratic_model(x_fit)
        plt.plot(x_fit, y_fit)

        # Calculate apex x-value: x = -b / (2a)
        x_apex = -b / (2 * a)
        apex_x_values[comparison_preids[i]] = x_apex
        total_previvor_windel[previd].append(x_apex)

plt.gca().invert_xaxis()
plt.xlabel('Days Before Cancer Diagnosis (Bins)', fontsize=20)
plt.ylabel('Jaccard Score', fontsize=20)
plt.title('Pre-vivor Cohort vs Comparison Cohort', fontsize=25)
plt.xlim(0, -3000)
plt.ylim(-.05, .25)
plt.grid(True)
plt.show()

1.5.2  Calculate Statistics and Plot the best WINDEL Scores for the Pre-Vivor Cohort

In [ ]:
best_previvor_windel = {}

for identification, w_scores in total_previvor_windel.items():
    w_scores = [t for t in w_scores if t < 0]
    validation_scores = []
    validation_scores.append(max(w_scores))
    validation_scores.append(np.mean(w_scores))
    validation_scores.append(np.median(w_scores))
    best_previvor_windel[identification] = validation_scores

previvor_best_scores = pd.DataFrame.from_dict(best_previvor_windel, orient='index', columns=['best_windel_score', 'mean_windel_score', 'median_windel_score'])
previvor_best_scores.reset_index(inplace=True)
previvor_best_scores.rename(columns={'index': 'id'}, inplace=True)

print(previvor_best_scores)

In [ ]:
previvor_best_scores['best_windel_score'].plot(kind='box')
plt.ylabel('Days before Cancer Diagnosis')
plt.title('Pre-Vivor Best Windel Scores')
plt.show()

# 1.6  Calculate WINDEL Scores for Test Breast Cancer Cohort

In [ ]:
indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg_test = {}

for rando1 in test_bc_cohort:
    for rando2 in comparison_cohort:
        person1_df = filtered_bc_binned[filtered_bc_binned['person_id'] == rando2]
        person2_df = filtered_bc_binned[filtered_bc_binned['person_id'] == rando1]

        person1_bin_set = set(person1_df['bin'])
        person2_bin_set = set(person2_df['bin'])

        bin_pairs = np.array([(bin1, bin2) for bin1 in person1_bin_set for bin2 in person2_bin_set], dtype=object)

        Asub_list = {bin1: set(person1_df.loc[person1_df['bin'] == bin1, 'concept_code']) for bin1 in person1_bin_set}
        Bsub_list = {bin2: set(person2_df.loc[person2_df['bin'] == bin2, 'concept_code']) for bin2 in person2_bin_set}
        similarity_scores = np.array([
            len(Asub_list[bin1] & Bsub_list[bin2]) / len(Asub_list[bin1] | Bsub_list[bin2]) if len(Asub_list[bin1] | Bsub_list[bin2]) > 0 else 0.0
            for bin1, bin2 in bin_pairs
        ])

        similarity_matrix = similarity_scores.reshape(len(person1_bin_set), len(person2_bin_set))
        mask = np.ones(similarity_matrix.shape, dtype=bool)
        max_dict_ordered = OrderedDict()
        excluded_bins_to_raw_score = OrderedDict()

        while np.any(mask):
            masked_matrix = np.where(mask, similarity_matrix, -np.inf)
            max_index = np.argmax(masked_matrix)
            max_row, max_col = np.unravel_index(max_index, similarity_matrix.shape)
            max_score = similarity_matrix[max_row, max_col]

            tuple_key = tuple(bin_pairs[max_row * len(person2_bin_set) + max_col])
            max_dict_ordered[tuple_key] = max_score
            excluded_bins_to_raw_score.setdefault(tuple_key, []).append(max_score)

            mask[max_row, :] = False
            mask[:, max_col] = False
        sorted_ordered_dict = OrderedDict(sorted(max_dict_ordered.items(), key=lambda x: x[1], reverse=True))
        indexes2actuallyplot.append(list(sorted_ordered_dict.keys()))
        averages_to_actually_graph = [st.mean(sorted_ordered_dict.values())]

        sorted_bins_to_max_score_dict = OrderedDict(sorted(excluded_bins_to_raw_score.items(), key=lambda x: max(x[1]), reverse=True))
        while sorted_bins_to_max_score_dict:
            bins_from_matrix = list(sorted_bins_to_max_score_dict.keys())
            maxes_average = st.mean([val for values in sorted_bins_to_max_score_dict.values() for val in values])
            averages_with_bins.setdefault((rando1, rando2), []).append({bins_from_matrix[-1]: maxes_average})
            sorted_bins_to_max_score_dict.popitem()

        person1_best_scores = {key[0]: value[0] for key, value in excluded_bins_to_raw_score.items()}
        person2_best_scores = {key[1]: value[0] for key, value in excluded_bins_to_raw_score.items()}

        sorted_by_bins_person1 = OrderedDict(sorted(person1_best_scores.items()))
        sorted_by_bins_person2 = OrderedDict(sorted(person2_best_scores.items()))

        if (rando2, rando1) not in split_avg_test:
            split_avg_test[(rando2, rando1)] = []

        while sorted_by_bins_person2:
            closest_to_diagnosis_p2 = next(iter(sorted_by_bins_person2))
            curr_average = st.mean(sorted_by_bins_person2.values())
            split_avg_test[(rando2, rando1)].append({closest_to_diagnosis_p2: curr_average})
            del sorted_by_bins_person2[closest_to_diagnosis_p2]

In [ ]:
result_bc_test = []
comparison_ids_test = [] # This is storing tuples
# the key is the person, the person is the dictionary of bins to scores
for key, value in split_avg_test.items():
    person2_bins = {}
    for val in value:
        for k, v in val.items():
            bin = int(k.split(',')[0])
            person2_bins[bin] = v
    scores = {}
    score_list = []
    curr_score = 0
    for j in bins:
        curr_bin = j
        if j not in person2_bins:
            curr_score = curr_score
        else:
            curr_score = person2_bins[j]
        scores[curr_bin] = curr_score
        score_list.append(curr_score)
    max_score = 0
    for item in score_list:
        if item > max_score:
            max_score = item
    # Is this a condition to remove duplicates? If so, move up in code
    if max_score != 1.0:
        result_bc_test.append(scores)
        comparison_ids_test.append(key)

In [ ]:
test_id_to_list_of_dictionaries = {}
for test_person in test_bc_cohort:
    test_id_to_list_of_dictionaries[test_person] = []
    for test_index in range(len(result_bc_test)):
        if comparison_ids_test[test_index][1] == test_person:
            test_id_to_list_of_dictionaries[test_person].append(result_bc_test[test_index])

1.6.1  Plot WINDEL Scores for Breast Cancer Test Cohort

In [ ]:
plt.figure(figsize=(20,14))

comparison_functions_test = {}
test_x_values = {}  # Optional: store apex x-values by ID
test_windel = {}

for test_id, test_group in test_id_to_list_of_dictionaries.items():
    test_windel[test_id] = []
    i_test = -1
    for dataset in test_group:
        i_test += 1
        x_test = np.array(list(dataset.keys()))
        y_test = np.array(list(dataset.values()))

        # Find the cutoff to ignore leading zero values
        index_test = 0
        while index_test < len(y_test) and y_test[index_test] == 0:
            index_test += 1
        y_cutoff_test = y_test[index_test:]
        x_cutoff_test = x_test[index_test:]
        if x_cutoff_test.size == 0:
            continue

        # Fit a quadratic model
        coefficients_test = np.polyfit(x_cutoff_test, y_cutoff_test, 2)  # coefficients = [a, b, c]
        a_test, b_test = coefficients_test[0], coefficients_test[1]
        comparison_functions_test[comparison_ids_test[i_test]] = coefficients_test

        # Create model and predictions
        quadratic_model_test = np.poly1d(coefficients_test)
        x_fit_test = np.linspace(x_test.min(), x_test.max(), 100)
        y_fit_test = quadratic_model_test(x_fit_test)
        plt.plot(x_fit_test, y_fit_test)

        # Calculate apex x-value: x = -b / (2a)
        x_apex_test = -b_test / (2 * a_test)
        test_x_values[comparison_ids_test[i_test]] = x_apex_test
        test_windel[test_id].append(x_apex_test)

plt.gca().invert_xaxis()
plt.xlabel('Days Before Cancer Diagnosis (Bins)', fontsize=20)
plt.ylabel('Jaccard Score', fontsize=20)
plt.title('Test vs Matched', fontsize=25)
plt.xlim(0, -3000)
plt.ylim(-.05, .25)
plt.grid(True)
plt.show()

1.6.2  Caluclate Statistics and Plot the Best WINDEL Scores for the Breast Cancer Test Cohort

In [ ]:
test_best_windel = {}

for identification, w_scores in test_windel.items():
    w_scores = [t for t in w_scores if t < 0]
    test_verification_scores = []
    test_verification_scores.append(max(w_scores))
    test_verification_scores.append(np.mean(w_scores))
    test_verification_scores.append(np.median(w_scores))
    test_best_windel[identification] = test_verification_scores

test_bc_best_scores = pd.DataFrame.from_dict(test_best_windel, orient='index', columns=['best_windel_score', 'mean_windel_score', 'median_windel_score'])
test_bc_best_scores.reset_index(inplace=True)
test_bc_best_scores.rename(columns={'index': 'id'}, inplace=True)

print(test_bc_best_scores)

In [ ]:
plt.figure(figsize=(8, 10))
test_bc_best_scores['best_windel_score'].plot(kind='box')
plt.ylabel('Days before Cancer Diagnosis')
plt.title('Breast Cancer Test Cohort Best Windel Scores')

plt.show()

# 1.7  Compare Pre-Vivor Cohort and Breast Cancer Test Cohort

In [ ]:
test_scores = test_bc_best_scores['best_windel_score'].tolist()
previvor_scores = previvor_best_scores['best_windel_score'].tolist()

data = [previvor_scores, test_scores]

plt.figure(figsize=(10, 12))
plt.boxplot(data, labels=["Pre-vivors", "Test"])
plt.ylabel("Days Before Cancer Diagnosis")
plt.title("Comparison of Best WINDEL Scores")
plt.grid(True)
plt.show()

In [ ]:
t_stat, p_value = ttest_ind(previvor_scores, test_scores)
print(f'T-Stat: {t_stat}')
print(f'P-Value: {p_value}')

# 1.8  Save Pre-Vivor and Breast Cancer Test Cohort Tables to the Workspace as .csv files

In [ ]:
# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

my_dataframe = previvor_best_scores

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename = 'previvor_statistics.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# save dataframe in a csv file in the same workspace as the notebook
my_dataframe.to_csv(destination_filename, index=False)

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file to the bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
output = subprocess.run(args, capture_output=True)

# print output from gsutil
output.stderr

In [ ]:
# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

my_dataframe = test_bc_best_scores

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename = 'test_cancer_statistics.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# save dataframe in a csv file in the same workspace as the notebook
my_dataframe.to_csv(destination_filename, index=False)

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file to the bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
output = subprocess.run(args, capture_output=True)

# print output from gsutil
output.stderr

# 1.9  Retrieve Previvor and Breast Cancer Test Cohort Tables from the Bucket

1.9.1  Grab Previvors

In [ ]:
# This snippet assumes you run setup first

# This code copies file in your Google Bucket and loads it into a dataframe

# Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
name_of_file_in_bucket = 'previvor_statistics.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
previvor_statistics = pd.read_csv(name_of_file_in_bucket)
previvor_statistics.head()

1.9.2  Grab Cancer Test Cohort

In [ ]:
# This snippet assumes you run setup first

# This code copies file in your Google Bucket and loads it into a dataframe

# Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
name_of_file_in_bucket = 'test_cancer_statistics.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
test_cancer_statistics = pd.read_csv(name_of_file_in_bucket)
test_cancer_statistics.head()

In [ ]:
best_parameter = "best_windel_score"
mean_parameter = "mean_windel_score"
median_parameter = "median_windel_score"

parameter = median_parameter

plt.plot(previvor_statistics[parameter])
plt.plot(test_cancer_statistics[parameter])
plt.legend()